# Classroom Compass — 04 · Report Builder

Builds the self-contained HTML insight report from a completed pipeline run.

**Prerequisites:** `report_template.html` and `chart.umd.min.js` in the project root.

**Sections**
1. Data Loading — run outputs + supplemental tables
2. User-Defined Functions — all UDFs consolidated here
3. Chart Data — per-insight demographic / resource distributions
4. Distinctness (TVD) — how distinct each insight is vs full population
5. Map Export — per-insight state distribution CSV
6. Report Builder — assemble JSON payload and render HTML

In [1]:
import sys, json, math, base64
from pathlib import Path

import pandas as pd
import numpy as np
import re as _re

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

# ── Load versioned utils module ───────────────────────────────────────────
import importlib.util as _ilu

_utils_path = next(Path(".").glob("utils*.py"))
if _utils_path.stem != "utils":
    _spec = _ilu.spec_from_file_location("utils", _utils_path)
    _mod  = _ilu.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    sys.modules["utils"] = _mod

from utils import load_cfg, resolve_params_path, apply_filters

CFG_PATH = resolve_params_path()
CFG      = load_cfg(CFG_PATH)

## 1 · Data Loading

In [2]:
# ── Run discovery ──────────────────────────────────────────────────────────
# Auto-discover run folders under OUTPUTS/runs/{strategic_area,non_strategic}.
# Keep only the latest timestamp per (intermediate folder, split). The split
# token is parsed from each run-folder name '<date>_<time>_<split>_<hash>'.
# For non_strategic the split equals the group folder; for strategic_area
# multiple splits can live under one area and are each kept independently.
RUNS_ROOT = Path("OUTPUTS/runs")

def _run_split(p):
    parts = p.name.split("_")
    return "_".join(parts[2:-1])          # drop date, time, and trailing hash

_candidates = [
    p
    for parent in ("strategic_area", "non_strategic")
    for p in (RUNS_ROOT / parent).glob("*/*")
    if (p / "metadata" / "pipeline_manifest.json").exists()
]
_latest = {}
for p in sorted(_candidates, key=lambda q: q.name):   # name is ts-prefixed → chronological
    _latest[(p.parent.name, _run_split(p))] = p        # newer overwrites older

ALL_RUNS = sorted(_latest.values(), key=lambda q: q.name)
if not ALL_RUNS:
    raise FileNotFoundError(
        f"No run folders found under {RUNS_ROOT}/{{strategic_area,non_strategic}}"
    )

# Primary run owns ANALYSIS_N, baselines, and the output folder. Prefer a
# non_strategic run so ANALYSIS_N reflects the full filtered corpus.
PRIMARY_RUN = next(
    (r for r in ALL_RUNS if r.parents[1].name == "non_strategic"), ALL_RUNS[0]
)
ADDITIONAL_RUNS = [r for r in ALL_RUNS if r != PRIMARY_RUN]

print(f"Discovered {len(ALL_RUNS)} run(s):")
for r in ALL_RUNS:
    tag = "[primary]" if r == PRIMARY_RUN else "         "
    print(f"  {tag} {r.parents[1].name}/{r.parent.name}/{r.name}")

# Manually define project cost bins; in empty [] then script will auto-choose
PROJ_COST_BINS = [0, 250, 500, 750, 1000, np.inf]
ENROLL_BINS    = [0, 300, 500, 800, 1000, np.inf]   # school_enrollment buckets: <300, 300-500, 500-800, 800-1000, 1000+

# ── Insight ranking / tier inputs (manually set paths) ────────────────────
RANKING_DIR   = Path("/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review")
RANKING_FILES = {
    "non_strategic":  RANKING_DIR / "insight_ranking_input_non_strategic.csv",
    "strategic_area": RANKING_DIR / "insight_ranking_input_strategic_area.csv",
}

# ── Display names for strategic areas / non-strategic groups ──────────────
# Folder name (run_path.parent.name) -> pretty label, used for the "Across X"
# cross-insight label and the group tabs. Unlisted folders fall back to
# title-case. Add overrides (esp. acronyms) as needed.
AREA_DISPLAY_NAMES = {
    "ai_usage_technology_literacy": "AI Usage & Technology Literacy",
}

def _display_name(folder):
    return AREA_DISPLAY_NAMES.get(str(folder), str(folder).replace("_", " ").title())

# ── Manifest + filter validation helpers ──────────────────────────────────
def _load_manifest(run_path):
    return json.loads((run_path / "metadata" / "pipeline_manifest.json").read_text())

def _load_filter_summary(run_path):
    p = run_path / "metadata" / "filter_summary.json"
    if not p.exists():
        raise FileNotFoundError(
            f"filter_summary.json missing from {run_path}. "
            "This file is required — re-run NB03 to regenerate it."
        )
    return json.loads(p.read_text())

def _validate_filter_compatibility(primary_manifest, primary_fsum, other_run_path):
    """Hard-fail if a supplemental run used different filters than the primary."""
    other_manifest = _load_manifest(other_run_path)
    other_fsum     = _load_filter_summary(other_run_path)

    pk = primary_manifest.get("filter_fields_key")
    ok = other_manifest.get("filter_fields_key")
    if pk != ok:
        raise ValueError(
            f"Filter key mismatch: primary={pk!r}, "
            f"{other_run_path.name}={ok!r}. "
            "All runs must use the same filter configuration."
        )

    pn = int(primary_fsum.get("output_row_count", 0))
    on = int(other_fsum.get("output_row_count", 0))
    if pn > 0 and on > 0:
        deviation_pct = abs(pn - on) / pn * 100
        if deviation_pct > 1.0:
            raise ValueError(
                f"Filter population mismatch: primary={pn:,}, "
                f"{other_run_path.name}={on:,} ({deviation_pct:.1f}% deviation). "
                "Runs must be generated from the same filtered population."
            )

# ── Run-aware ID prefix ───────────────────────────────────────────────────
def _run_prefix(manifest, run_path):
    """Unique per-run namespace. run_path.name is '<date>_<time>_<split>_<hash>';
    its trailing hash + the parent folder (strategic area, or non-strategic group)
    uniquely identify the run. The old run_id[:8] was date-only, so distinct
    insights from runs sharing (group_by_field, date) collided onto one id."""
    parent   = run_path.parent.name
    run_hash = run_path.name.split("_")[-1]
    return f"{manifest['group_by_field']}__{parent}__{run_hash}"

def _prefix_ids(df, prefix):
    df = df.copy()
    df["insight_id"] = prefix + "__" + df["insight_id"].astype(str)
    return df

# ── Load primary run ──────────────────────────────────────────────────────
_primary_manifest = _load_manifest(PRIMARY_RUN)
_primary_fsum     = _load_filter_summary(PRIMARY_RUN)

RUN_ID            = _primary_manifest["run_id"]
RUN_DATE          = _primary_manifest["run_date"]
GROUPBY_FIELD     = _primary_manifest["group_by_field"]   # primary field; kept for single-run compat
FILTER_FIELDS_KEY = _primary_manifest["filter_fields_key"]
ANALYSIS_N        = int(_primary_fsum["output_row_count"])

def OUT(subdir, fname):
    """Write outputs to the primary run folder."""
    p = PRIMARY_RUN / subdir
    p.mkdir(parents=True, exist_ok=True)
    return p / fname

# ── Validate + merge all runs ─────────────────────────────────────────────
_all_run_prefixes = []
insight_project_frames, curated_frames, insights_flat_frames = [], [], []
structured_merged = {"key_insights": [], "by_group": {}}

for run_path in ALL_RUNS:
    manifest   = _load_manifest(run_path)
    fsum       = _load_filter_summary(run_path)
    groupby    = manifest["group_by_field"]
    run_id_str = str(manifest["run_id"])

    if run_path != PRIMARY_RUN:
        try:
            _validate_filter_compatibility(_primary_manifest, _primary_fsum, run_path)
        except ValueError as e:
            print(f"  [filter-check warning] {e}")

    prefix = _run_prefix(manifest, run_path)
    _all_run_prefixes.append(prefix)

    ip_df = _prefix_ids(pd.read_csv(run_path / "insights"   / "insight_project_bridge.csv"), prefix)
    cu_df = _prefix_ids(pd.read_csv(run_path / "chart_data" / "curated_df.csv"),             prefix)
    fl_df = _prefix_ids(pd.read_csv(run_path / "insights"   / "insights_flat.csv"),          prefix)

    for frame in (ip_df, cu_df, fl_df):
        frame["source_groupby_field"] = groupby
        frame["source_run_id"]        = run_id_str
        frame["source_run_type"]      = run_path.parents[1].name   # strategic_area | non_strategic
        frame["source_parent"]        = run_path.parent.name       # area folder / group folder

    insight_project_frames.append(ip_df)
    curated_frames.append(cu_df)
    insights_flat_frames.append(fl_df)

    # Merge structured JSON. Namespace both insight_id and by_group bucket keys
    # so that "Books" from project_category and "Other" from metro_type don't collide.
    raw = json.loads((run_path / "insights" / "insights_structured.json").read_text())
    for item in raw.get("key_insights", []):
        item = dict(item)
        item["insight_id"]           = f"{prefix}__{item['insight_id']}"
        item["source_groupby_field"] = groupby
        item["source_run_id"]        = run_id_str
        structured_merged["key_insights"].append(item)
    for group_val, items in raw.get("by_group", {}).items():
        bucket_key = f"{groupby}::{group_val}"          # namespaced bucket
        prefixed = []
        for i in items:
            i = dict(i)
            i["insight_id"]           = f"{prefix}__{i['insight_id']}"
            i["source_groupby_field"] = groupby
            i["source_run_id"]        = run_id_str
            prefixed.append(i)
        structured_merged["by_group"].setdefault(bucket_key, []).extend(prefixed)

insight_project_df = pd.concat(insight_project_frames, ignore_index=True)
curated_df         = pd.concat(curated_frames,         ignore_index=True)
insights_flat_df   = pd.concat(insights_flat_frames,   ignore_index=True)
structured         = structured_merged

# Namespace category_bucket in curated_df to match the structured bucket keys above.
# This keeps by-group grouping semantically coherent across lenses.
curated_df["category_bucket"] = (
    curated_df["source_groupby_field"] + "::" + curated_df["category_bucket"].astype(str)
)

# ── Ask Compass: attach global_insight_id ────────────────────────────────
# The Ask Compass registry is keyed on global_insight_id, produced by the
# cc_ union path. Rather than recompute that recipe here (its docstring
# warns it must stay byte-identical to the tier union script, and the union
# enriches strategic_area_id from run manifests first), join to the ids the
# cc_ report already published.
#
# Join key is (source_run_id, title): verified unique across all report rows.
# Unmatched rows keep their NB05 id and simply will not highlight, which is
# correct for insights the frozen agent snapshot does not contain.

_cc_reports = sorted(
    (ROOT / "OUTPUTS" / "sweep_review" / "reports").glob(
        "classroom_compass_report_data_*.json")
)

curated_df["global_insight_id"] = ""
_ac_matched = 0

if not _cc_reports:
    print("[ask-compass] no cc_ report data found — cards will not highlight")
else:
    _cc = json.loads(_cc_reports[-1].read_text(encoding="utf-8"))
    _gid_map = {
        (str(i.get("source_run_id", "")), str(i.get("title", "")).strip()):
            str(i.get("global_insight_id") or i.get("id") or "")
        for i in _cc.get("insights", [])
    }

    _keys = list(zip(
        curated_df["source_run_id"].astype(str),
        curated_df["title"].astype(str).str.strip(),
    ))
    curated_df["global_insight_id"] = [_gid_map.get(k, "") for k in _keys]
    _ac_matched = int((curated_df["global_insight_id"] != "").sum())

    # A duplicate global id would make two cards share a DOM id and break
    # both. Drop the later one back to its NB05 id rather than collide.
    _dupes = curated_df.loc[curated_df["global_insight_id"] != "",
                            "global_insight_id"].duplicated()
    if _dupes.any():
        curated_df.loc[_dupes[_dupes].index, "global_insight_id"] = ""
        print(f"[ask-compass] {int(_dupes.sum())} duplicate global id(s) "
              "dropped back to NB05 ids")
        _ac_matched = int((curated_df["global_insight_id"] != "").sum())

    print(f"[ask-compass] source: {_cc_reports[-1].name}")
    print(f"[ask-compass] matched {_ac_matched:,} of {len(curated_df):,} "
          f"insights to global_insight_id")
    _unmatched = len(curated_df) - _ac_matched
    if _unmatched:
        print(f"[ask-compass] {_unmatched} insight(s) not in the agent "
              "snapshot; those cards will not highlight")
        for _t in curated_df.loc[curated_df["global_insight_id"] == "",
                                 "title"].head(5):
            print(f"               · {_t}")

# ── Shared corpus — filtered to ANALYSIS_N scope ─────────────────────────
# Apply the same filters NB03 used so all baselines reflect the analysis
# population, not the full unfiltered parquet.
df = pd.read_parquet(ROOT / "OUTPUTS/prepared/06_enriched.parquet")

# Merge first so 'state' exists when apply_filters runs
project_attributes_df = (
    pd.read_csv(ROOT / "DATA/project_attributes.csv", usecols=["project_id", "state"])
    .drop_duplicates("project_id")   # prevents row explosion
)

# Merge first so 'state' exists when apply_filters runs.
# The enriched parquet may already carry 'state'; only pull it from
# project_attributes.csv when it's missing, to avoid a state_x/state_y collision.
if "state" not in df.columns:
    project_attributes_df = (
        pd.read_csv(ROOT / "DATA/project_attributes.csv", usecols=["project_id", "state"])
        .drop_duplicates("project_id")   # prevents row explosion
    )
    df = df.merge(project_attributes_df, on="project_id", how="left")

# Ensure teacher_is_teacher_of_color is available for its chart + filter
if "teacher_is_teacher_of_color" not in df.columns:
    _toc = (
        pd.read_csv(ROOT / "DATA/project_attributes.csv",
                    usecols=["project_id", "teacher_is_teacher_of_color"])
        .drop_duplicates("project_id")
    )
    df = df.merge(_toc, on="project_id", how="left")

_filters_list = CFG.get("analysis", {}).get("filters", [])
_filter_logic = CFG.get("analysis", {}).get("filter_logic", "and")
if _filters_list:
    df, _fsum_check = apply_filters(df, _filter_logic, _filters_list)
    _dev = abs(len(df) - ANALYSIS_N) / max(ANALYSIS_N, 1)
    if _dev >= 0.01:
        print(f"[filter-check warning] Filtered df ({len(df):,}) deviates "
              f"{_dev*100:.1f}% from ANALYSIS_N ({ANALYSIS_N:,}); expected when the "
              "primary run is corpus-scoped but some runs are area-scoped.")

resource_category_df  = pd.read_csv(ROOT / "DATA/project_resource_categorylevel.csv")
resource_item_df      = pd.read_csv(ROOT / "DATA/project_resource_itemlevel.csv")

ALL_GROUPBY_FIELDS = list(dict.fromkeys(
    _load_manifest(r)["group_by_field"] for r in ALL_RUNS
))

print(f"Primary run:         {RUN_ID}")
print(f"ANALYSIS_N:          {ANALYSIS_N:,}")
print(f"Lenses (groupby):    {ALL_GROUPBY_FIELDS}")
print(f"Runs merged:         {len(ALL_RUNS)}")
print(f"ID prefixes:         {_all_run_prefixes}")
print(f"df:                  {len(df):,} rows")
print(f"insight_project_df:  {len(insight_project_df):,} rows")
print(f"curated_df:          {len(curated_df):,} rows")
print(f"insights_flat_df:    {len(insights_flat_df):,} rows")

Discovered 60 run(s):
            strategic_area/classroom_and_basic_needs/20260522_211311_strategic_injected_tag_f79bb1ee
            strategic_area/classroom_and_basic_needs/20260522_212017_project_category_1a49c96e
            strategic_area/mental_health_sel/20260522_212558_strategic_injected_tag_84753f13
            strategic_area/mental_health_sel/20260522_213028_project_category_2172bad3
            strategic_area/stem/20260522_213459_strategic_injected_tag_b09019b8
            strategic_area/stem/20260522_213844_project_category_33a560da
            strategic_area/workforce_development/20260522_214306_strategic_injected_tag_378072a6
            strategic_area/workforce_development/20260522_214713_project_category_16003543
            strategic_area/industry/20260522_215223_strategic_injected_tag_0cc3ff3a
            strategic_area/industry/20260522_215603_project_category_85099d30
            strategic_area/specialty_subject/20260522_220016_strategic_injected_tag_871a8409
     

## 2 · User-Defined Functions

In [3]:
# =============================================================================
# All UDFs for chart data prep, TVD, and report building.
# Imported utilities (load_cfg, resolve_params_path) live in utils.py.
# =============================================================================

# ── Chart data helpers ────────────────────────────────────────────────────

def counts_pct(s):
    """Value counts with proportions for a Series.
    Returns DataFrame with columns [category, count, pct]."""
    c = s.value_counts(dropna=False).rename_axis("category").reset_index(name="count")
    c["pct"] = (c["count"] / c["count"].sum()).round(4)
    return c

def topN_counts_pct(s, n=10):
    """Top-N categorical values by frequency.
    Returns DataFrame with columns [category, count, pct]."""
    c = counts_pct(s)
    if c.empty:
        return pd.DataFrame(columns=["category", "count", "pct"])
    return (
        c.sort_values(["count", "category"], ascending=[False, True])
         .head(n)
         .reset_index(drop=True)
    )

def quintile_bins(df, col, label, bins=None):
    """Bin a numeric column into quintiles; reuse thresholds when bins is provided.
    Returns (result_df, bins) where result_df has [min, max, count, pct]."""
    s = df[col].dropna()
    if bins is None:
        cut, bins = pd.qcut(s, 5, retbins=True, duplicates="drop")
    else:
        cut = pd.cut(s, bins=bins, include_lowest=True)
    result = cut.value_counts().sort_index().reset_index()
    result.columns = [label, "count"]
    result["pct"] = (result["count"] / result["count"].sum()).round(4)
    result["min"] = [iv.left  for iv in result[label].cat.categories]
    result["max"] = [iv.right for iv in result[label].cat.categories]
    return result[["min", "max", "count", "pct"]], bins

def fy_from_date(s):
    """Return fiscal year string (Jul-start) from a date Series. E.g. FY24, FY25."""
    s = pd.to_datetime(s)
    def _fy(d):
        if pd.isna(d): return None
        return f"FY{(d.year + 1) % 100:02d}" if d.month >= 7 else f"FY{d.year % 100:02d}"
    return s.apply(_fy)

def fy_half(s):
    """Return H1 (Jul–Dec) or H2 (Jan–Jun) from a date Series."""
    return pd.to_datetime(s).dt.month.apply(
        lambda m: "H1 (Jul-Dec)" if m >= 7 else "H2 (Jan-Jun)"
    )

def assign_efs(row):
    """Return the EFS classification string for a project row."""
    rural = row["school_is_underserved_rural"]                    == "Yes"
    race  = row["school_is_historically_underrepresented_race"]   == "Yes"
    inc   = row["school_is_low_income"]                           == "Yes"
    if race and inc: return "Race+Inc"
    if rural:        return "Rural"
    if race:         return "Race"
    if inc:          return "Inc"
    return "NonEFS"

def topN_categories(df_cat, n=10):
    """Top-N item categories by total quantity for a resource_category subset.
    Returns DataFrame with [category, quantity_count, pct]. Default n=10."""
    if df_cat.empty:
        return pd.DataFrame(columns=["category", "quantity_count", "pct"])
    top = (
        df_cat.groupby("item_category", dropna=True)["quantity_count"]
        .sum().nlargest(n).reset_index()
        .rename(columns={"item_category": "category"})
    )
    total = df_cat["quantity_count"].sum()
    top["pct"] = (top["quantity_count"] / total).round(4) if total > 0 else 0.0
    return top[["category", "quantity_count", "pct"]]                .sort_values("quantity_count", ascending=False)                .reset_index(drop=True)

# ── TVD helpers ───────────────────────────────────────────────────────────

def tvd(df_a, df_b, key_col, val_col="pct"):
    """Total variation distance between two categorical distributions."""
    a = df_a[[key_col, val_col]].assign(**{key_col: df_a[key_col].astype(str)})
    b = df_b[[key_col, val_col]].assign(**{key_col: df_b[key_col].astype(str)})
    m = a.merge(b, on=key_col, how="outer", suffixes=("_a","_b")).fillna(0)
    return round((m[f"{val_col}_a"] - m[f"{val_col}_b"]).abs().sum() / 2, 4)

def race_dist(df_a, df_b):
    """Sum of absolute race-distribution differences, normalised to 0–1."""
    m = df_a.merge(df_b, on="race", suffixes=("_a","_b"))
    return round((m["weighted_avg_pct_a"] - m["weighted_avg_pct_b"]).abs().sum() / 100, 4)

# ── Report-builder helpers ────────────────────────────────────────────────

# Build a compact date-range string from the active filters for the header chip
def _filter_range_str(filters):
    parts = []
    for f in filters:
        if f.get('op') == 'range':
            lo = f.get('min', '')
            hi = f.get('max') or 'present'
            parts.append(f"{lo} to {hi}")
        elif f.get('op') in ('eq', 'in'):
            val = f.get('value') or ', '.join(str(v) for v in f.get('values', []))
            parts.append(str(val))
    return '  ·  '.join(parts) if parts else 'All time'

# Fields included in the detail-panel charts (order matters for display)
DETAIL_FIELDS = ["metro", "grade", "efs", "race", "item_category", "state", "posting", "cost_distr", "funding", "project_category", "teacher_of_color", "enroll_distr"]

EFS_ORDER  = ["Race+Inc", "Rural", "Race", "Inc", "NonEFS"]

RACE_COLS  = {
    "Black":  "school_percent_black_imputed",
    "Latinx": "school_percent_latinx_imputed",
    "Asian":  "school_percent_asian_imputed",
    "White":  "school_percent_white_imputed",
}

FIELD_META = {
    "metro":         {"val_col": "pct",              "sort_col": "category"},
    "grade":         {"val_col": "pct",              "sort_col": "category"},
    "efs":           {"val_col": "pct",              "sort_col": "category"},
    "race":          {"val_col": "weighted_avg_pct", "sort_col": "race"},
    "item_category": {"val_col": "pct",   "sort_col": "quantity_count", "label_col": "category"},
    "state":         {"val_col": "pct",              "sort_col": "category"},
    "posting":       {"val_col": "pct",   "sort_col": "period_sort",    "label_col": "category"},
    "cost_distr":    {"val_col": "pct",   "sort_col": "min",            "label_col": "cost_label"},
    "enroll_distr":  {"val_col": "pct",   "sort_col": "min",            "label_col": "enroll_label"},
    "funding":       {"val_col": "pct", "sort_col": "category"},
    "project_category": {"val_col": "pct", "sort_col": "category"},
    "teacher_of_color": {"val_col": "pct", "sort_col": "category"},
}

SEP_SUFFIX = {
    "metro": "school coverage",
    "grade": "coverage",
    "efs":   "school coverage",
    "race":  "student population",
}

def _safe_float(v):
    """Convert v to float; return 0.0 for NaN, Inf, or unconvertible values."""
    try:
        f = float(v)
        return 0.0 if math.isnan(f) or math.isinf(f) else f
    except (TypeError, ValueError):
        return 0.0

def df_to_chart(df, field):
    """Convert a chart DataFrame to (labels, values) lists using FIELD_META config."""
    meta      = FIELD_META[field]
    val_col   = meta["val_col"]
    sort_col  = meta["sort_col"]
    label_col = meta.get("label_col", sort_col)
    df_s = df.copy()
    if field == "efs":
        order_map = {v: i for i, v in enumerate(EFS_ORDER)}
        df_s["_ord"] = df_s["category"].map(order_map).fillna(99)
        df_s = df_s.sort_values("_ord").reset_index(drop=True)
    else:
        df_s = df_s.sort_values(sort_col).reset_index(drop=True)
    labels = df_s[label_col].astype(str).tolist()
    values = [_safe_float(v) for v in df_s[val_col]]
    return labels, values

def compute_separators(ins_charts, fc_charts_local):
    """Find top-2 field+value combos where this insight is most above-average vs corpus.
    Race values are on 0–100 scale; others on 0–1. Both converted to ppt for comparison."""
    candidates = []
    for field in DETAIL_FIELDS:
        if field == "teacher_of_color":
            continue
        ch = ins_charts.get(field)
        fc = fc_charts_local.get(field)
        if not ch or not fc:
            continue
        is_race = field == "race"
        fc_map  = dict(zip(fc["labels"], fc["values"]))
        for label, val in zip(ch["labels"], ch["values"]):
            diff_ppt = (val - fc_map.get(label, 0)) * (1 if is_race else 100)
            if diff_ppt > 0.5:
                candidates.append({
                    "field":    field,
                    "label":    label,
                    "diff_ppt": diff_ppt,
                    "suffix":   SEP_SUFFIX.get(field, ""),
                })
    candidates.sort(key=lambda x: x["diff_ppt"], reverse=True)
    result = {}
    for n, key in enumerate(["sep1", "sep2"], 1):
        if len(candidates) >= n:
            c = candidates[n - 1]
            result[f"{key}_val"] = round(c["diff_ppt"], 0)
            result[f"{key}_lbl"] = f"More {c['label']} {c['suffix']}".strip()
        else:
            result[f"{key}_val"] = None
            result[f"{key}_lbl"] = None
    return result

def _extract_body(d):
    """Extract prose fields from a structured insight dict."""
    return {
        "finding":         str(d.get("finding")         or "").strip(),
        "evidence_basis":  str(d.get("evidence_basis")  or "").strip(),
        "scope_or_caveat": str(d.get("scope_or_caveat") or "").strip(),
        "why_it_matters":  str(d.get("why_it_matters")  or "").strip(),
    }


def _walk(obj, out):
    """Recursively walk structured JSON and populate out[insight_id] with body text."""
    if isinstance(obj, dict):
        if "insight_id" in obj and obj["insight_id"] not in out:
            out[obj["insight_id"]] = _extract_body(obj)
        for v in obj.values():
            _walk(v, out)
    elif isinstance(obj, list):
        for item in obj:
            _walk(item, out)

def _is_cross(row):
    """Return True if this curated_df row represents a cross-category insight."""
    rs = str(row.get("report_section", "")).lower()
    s  = str(row.get("section",        "")).lower()
    return ("cross" in rs or s == "key_insights"
            or not row.get("category_bucket")
            or str(row.get("category_bucket", "")).lower() in ("none", ""))

def _fmt_filter(f):
    """Format a single CFG filter dict as a human-readable string."""
    field = f.get("field", "?")
    op    = f.get("op", "?")
    if op == "eq":
        return f"{field} = {f.get('value', '?')}"
    if op == "in":
        vals = f.get("values", f.get("value", "?"))
        if isinstance(vals, list):
            extra = f" (+{len(vals)-5} more)" if len(vals) > 5 else ""
            vals  = ", ".join(str(v) for v in vals[:5]) + extra
        return f"{field} in [{vals}]"
    if op == "range":
        lo = f.get("min", "?")
        hi = f.get("max") or "present"
        return f"{field} from {lo} to {hi}"
    if op == "not_null": return f"{field} is present"
    if op == "is_null":  return f"{field} is absent"
    return f"{field} {op}"

def _load_insight_tiers(ranking_files):
    """Load each ranking CSV, quartile composite_avg WITHIN each file (A = best),
    then union. Returns {(source, normalized_title): tier}. Quartiles are computed
    per file (rank-then-qcut so ties don't collapse bins), never on the union."""
    frames = []
    for source, path in ranking_files.items():
        if not Path(path).exists():
            print(f"  [tier warning] ranking file not found: {path}")
            continue
        t = pd.read_csv(path)
        t = t[t["composite_avg"].notna()].copy()
        if t.empty:
            continue
        ranks = t["composite_avg"].rank(method="first")          # ascending: low -> high
        t["tier"]       = pd.qcut(ranks, 4, labels=["D", "C", "B", "A"]).astype(str)
        t["_source"]    = source
        t["_title_key"] = t["title"].astype(str).str.strip().str.casefold()
        frames.append(t[["_source", "_title_key", "tier"]])
    if not frames:
        return {}
    allt = pd.concat(frames, ignore_index=True)
    # On the rare duplicate (source, title), keep the best tier (A < B alphabetically).
    allt = allt.sort_values("tier").drop_duplicates(["_source", "_title_key"], keep="first")
    return dict(zip(zip(allt["_source"], allt["_title_key"]), allt["tier"]))

## 3 · Chart Data

In [4]:
# ── Derived columns (computed once on the per-project frame) ───────────────
# These ride through the bridge merge below, so they are never recomputed on
# the exploded (insight x project) frame.
df["efs_category"] = df.apply(assign_efs, axis=1)
df["_fy"]          = fy_from_date(df["posted_date"])
df["_half"]        = fy_half(df["posted_date"])
df["_exp_years"]   = (pd.to_datetime(df["posted_date"]).dt.year
                      - df["teacher_start_teaching_year"])

df_full = df

# ── Enrich insight-project bridge (derived columns come through the merge) ──
df_enriched = insight_project_df.merge(df, on="project_id", how="left")

# ── Resource joins ────────────────────────────────────────────────────────
cat_enriched  = insight_project_df.merge(resource_category_df, on="project_id", how="inner")
item_enriched = insight_project_df.merge(resource_item_df,     on="project_id", how="inner")

# ── Top-5 resource category table ────────────────────────────────────────
cat_rows = []
for iid, grp in cat_enriched.groupby("insight_id"):
    topN = topN_categories(grp, n=6)
    topN.insert(0, "rank",       list(range(1, len(topN) + 1)))
    topN.insert(0, "insight_id", iid)
    cat_rows.append(topN)
chart_ready_category_df = (
    pd.concat(cat_rows, ignore_index=True)
    [["insight_id","rank","category","quantity_count","pct"]]
)
chart_ready_category_df.to_csv(OUT("chart_data", "chart_ready_category.csv"), index=False)

# ── Top-10 item names table ───────────────────────────────────────────────
item_rows = []
for iid, grp in item_enriched.groupby("insight_id"):
    top10 = (
        grp.groupby("item_name", dropna=True)["quantity_count"]
        .sum().nlargest(10).reset_index()
    )
    top10.insert(0, "rank",       range(1, len(top10) + 1))
    top10.insert(0, "insight_id", iid)
    item_rows.append(top10)
chart_ready_items_df = (
    pd.concat(item_rows, ignore_index=True)
    [["insight_id","rank","item_name","quantity_count"]]
)
chart_ready_items_df.to_csv(OUT("chart_data", "chart_ready_items.csv"), index=False)

# ── Build insight_charts dict ─────────────────────────────────────────────
# Keys: 'full_corpus', 'total_sample', <insight_id>, ...
# Each value: dict of field → DataFrame

_cat_lookup = {iid: grp for iid, grp in cat_enriched.groupby("insight_id")}
insight_charts = {}
cost_bins   = PROJ_COST_BINS if PROJ_COST_BINS else None
exp_bins    = None
enroll_bins = ENROLL_BINS if ENROLL_BINS else None

CHART_SOURCES = (
    [("full_corpus", df_full), ("total_sample", df_enriched)]
    + list(df_enriched.groupby("insight_id"))
)

for insight_id, df_i in CHART_SOURCES:
    enroll = df_i["school_enrollment"]

    # Cost & experience: bin thresholds locked to full_corpus on first pass
    cost_result, cost_bins = quintile_bins(
        df_i.dropna(subset=["total_cost"]), "total_cost", "cost_bucket", bins=cost_bins
    )
    cost_result["cost_label"] = cost_result.apply(
    lambda r: f"${r['min']:,.0f}+" if np.isinf(r["max"])
              else f"${r['min']:,.0f}–${r['max']:,.0f}", axis=1
    )
    enroll_result, enroll_bins = quintile_bins(
        df_i.dropna(subset=["school_enrollment"]), "school_enrollment", "enroll_bucket", bins=enroll_bins
    )
    enroll_result["enroll_label"] = enroll_result.apply(
        lambda r: f"{r['min']:,.0f}+" if np.isinf(r["max"])
                  else (f"<{r['max']:,.0f}" if r["min"] == 0
                        else f"{r['min']:,.0f}–{r['max']:,.0f}"), axis=1
    )
    exper_result, exp_bins = quintile_bins(
        df_i.dropna(subset=["_exp_years"]), "_exp_years", "experience_bucket", bins=exp_bins
    )

    # Posting period with chronological sort key
    posting = (
        df_i.assign(
            half_short=df_i["_half"].str.extract(r"^(H[12])", expand=False),
            fy_num=pd.to_numeric(
                df_i["_fy"].str.replace("FY", "", regex=False), errors="coerce"
            ),
        )
        .groupby(["_fy","half_short","fy_num"], dropna=False)
        .size().reset_index(name="count")
        .rename(columns={"_fy":"fy"})
    )
    posting["category"]    = posting["fy"] + " " + posting["half_short"]
    posting["period_sort"] = (posting["fy_num"] * 10
                              + posting["half_short"].map({"H1":1,"H2":2}).fillna(0))
    posting = posting[posting["fy_num"].notna()].copy()
    posting["pct"] = (posting["count"] / posting["count"].sum()).round(4)

    # State distribution (2-letter abbreviations)
    state_dist = counts_pct(df_i["state"]) if "state" in df_i.columns         else pd.DataFrame(columns=["category","count","pct"])

    # Resource categories for full_corpus and total_sample
    if insight_id == "full_corpus":
        cat_df = resource_category_df
    elif insight_id == "total_sample":
        cat_df = cat_enriched
    else:
        cat_df = _cat_lookup.get(insight_id, pd.DataFrame(columns=["item_category","quantity_count"]))

    insight_charts[insight_id] = {
        "project_count": len(df_i),
        "cost_distr":    cost_result,
        "enroll_distr":  enroll_result,
        "posting":       posting,
        "exper_distr":   exper_result,
        "metro":         counts_pct(df_i["metro_type_at_time_of_posting"]),
        "grade":         counts_pct(df_i["grade_band"]),
        "efs":           counts_pct(df_i["efs_category"]),
        "funding":       counts_pct(pd.Series(
            np.select(
                [df_i["funded_date"].notna(),
                 pd.to_datetime(df_i.get("expiration_date", pd.Series(dtype=str)),
                                errors="coerce") <= pd.Timestamp.today()],
                ["Funded", "Expired"],
                default="Live"
            ), index=df_i.index
        )),
        "race": pd.DataFrame([
            {
                "race": r,
                "weighted_avg_pct": round(
                    (df_i[col] * enroll).sum() / enroll.sum(), 4
                ) if enroll.sum() > 0 else 0.0
            }
            for r, col in RACE_COLS.items()
            if col in df_i.columns
        ]),
        "item_category": topN_categories(cat_df, n=6),
        "state":         state_dist,
        "teacher_of_color": (
            counts_pct(df_i["teacher_is_teacher_of_color"])
            if "teacher_is_teacher_of_color" in df_i.columns
            else pd.DataFrame(columns=["category", "count", "pct"])
        ),
        "project_category": (
            counts_pct(df_i["project_category"])
            if "project_category" in df_i.columns and insight_id == "full_corpus"
            else topN_counts_pct(df_i["project_category"], n=6)
            if "project_category" in df_i.columns
            else pd.DataFrame(columns=["category","count","pct"])
        ),
    }

print(f"Built chart data for {len(insight_charts)} entries "
      f"(incl. full_corpus + total_sample)")

Built chart data for 1054 entries (incl. full_corpus + total_sample)


## 4 · Distinctness (TVD)

In [5]:
# Measures how distinct each insight's distributions are vs the full corpus.
# Only insights with > 500 supporting projects are included.

corpus  = insight_charts["full_corpus"]
total_n = insight_charts["total_sample"]["project_count"]
tvd_cols = ["funding", "cost_distr", "exper_distr", "metro", "grade", "efs", "race"]

TVD_KEY = {
    "funding": "category", "cost_distr": "min", "exper_distr": "min",
    "metro": "category",   "grade": "category", "efs": "category",
    "race": "race",
}

distinctness = []
for insight_id, charts in insight_charts.items():
    if insight_id in ("full_corpus", "total_sample"):
        continue
    row = {
        "insight_id":    insight_id,
        "project_count": charts["project_count"],
        "pct_of_total":  round(charts["project_count"] / total_n, 4),
    }
    for field in tvd_cols:
        key = TVD_KEY[field]
        if field == "race":
            row[field] = race_dist(charts["race"], corpus["race"])
        else:
            row[field] = tvd(charts[field], corpus[field], key)
    distinctness.append(row)

distinctness_df = (
    pd.DataFrame(distinctness)
    .pipe(lambda d: d[d["project_count"] > 500])
    .reset_index(drop=True)
)
distinctness_df["mean_tvd"] = (
    distinctness_df[tvd_cols].mean(axis=1).round(4)
)
distinctness_df = (
    distinctness_df.sort_values("mean_tvd", ascending=False)
    .reset_index(drop=True)
)

print(f"Distinctness table: {len(distinctness_df)} insights with > 500 projects")
distinctness_df.head(10)

Distinctness table: 752 insights with > 500 projects


,insight_id,project_count,pct_of_total,funding,cost_distr,exper_distr,metro,grade,efs,race,mean_tvd
0,project_category_x_urbanity__project_category_...,706,0.0003,0.1613,0.2203,0.1219,0.8826,0.4114,0.4584,0.6753,0.4187
1,project_category_x_urbanity__project_category_...,536,0.0003,0.1643,0.1893,0.0401,0.8826,0.0868,0.5376,0.7527,0.3791
2,project_category_x_urbanity__project_category_...,1328,0.0006,0.0402,0.0523,0.0456,0.8826,0.3832,0.4311,0.5855,0.3458
3,grade_band_x_urbanity__grade_band_x_urbanity__...,561,0.0003,0.0564,0.2221,0.3207,0.9457,0.6295,0.1085,0.1016,0.3406
4,project_category_x_urbanity__project_category_...,682,0.0003,0.0402,0.1706,0.0479,0.8826,0.1170,0.4444,0.6382,0.3344
5,project_category_x_low_income_binary__project_...,578,0.0003,0.0929,0.4296,0.1320,0.1614,0.3369,0.6372,0.5367,0.3324
6,project_category_x_underserved_rural_binary__p...,573,0.0003,0.1091,0.0920,0.0654,0.6608,0.1915,0.5828,0.6069,0.3298
7,metro_type_at_time_of_posting__metro_type_at_t...,2085,0.0010,0.1126,0.1502,0.1074,0.9336,0.1264,0.3148,0.5550,0.3286
8,project_category_x_urbanity__project_category_...,522,0.0003,0.1491,0.1581,0.0248,0.9336,0.2225,0.2226,0.4794,0.3129
9,project_category_x_urbanity__project_category_...,1304,0.0006,0.0630,0.0516,0.0708,0.8818,0.1526,0.4063,0.5400,0.3094


## 5 · Map Export

In [6]:
# Per-insight project count keyed by 2-letter state abbreviation.
# The 'full_corpus' row gives the baseline distribution.

map_rows = []

for insight_id, df_i in [("full_corpus", df_full)] + list(df_enriched.groupby("insight_id")):
    if "state" not in df_i.columns:
        continue
    counts = (
        df_i["state"]
        .value_counts(dropna=True)
        .rename_axis("state")
        .reset_index(name="project_count")
    )
    total = counts["project_count"].sum()
    counts["pct"] = (counts["project_count"] / total).round(4)
    counts.insert(0, "insight_id", insight_id)
    map_rows.append(counts)

if map_rows:
    map_df = pd.concat(map_rows, ignore_index=True)
    map_df.to_csv(OUT("chart_data", "chart_ready_map.csv"), index=False)
    print(f"Map table: {len(map_df):,} rows | "
          f"{map_df['state'].nunique()} states | "
          f"{map_df['insight_id'].nunique()} insights (incl. full_corpus)")
    display(map_df[map_df["insight_id"] == "full_corpus"].head(10))
else:
    print("WARNING: 'state' column not found — check project_attributes.csv join in Section 1.")

Map table: 44,778 rows | 51 states | 1053 insights (incl. full_corpus)


,insight_id,state,project_count,pct
0,full_corpus,CA,110133,0.1345
1,full_corpus,TX,65357,0.0798
2,full_corpus,NY,65247,0.0797
3,full_corpus,FL,39434,0.0482
4,full_corpus,CO,35426,0.0433
5,full_corpus,NV,34448,0.0421
6,full_corpus,IL,30479,0.0372
7,full_corpus,OK,29619,0.0362
8,full_corpus,HI,24733,0.0302
9,full_corpus,PA,24294,0.0297


## 6 · Report Builder

In [7]:
# Reads outputs from Sections 3–5 and writes a self-contained HTML report.

# ── Emoji favicon ─────────────────────────────────────────────────────────
_svg = '''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64">
  <circle cx="32" cy="32" r="30" fill="#FBBC04"/>
  <polygon points="32,10 46,54 32,44 18,54" fill="#3E00C9"/>
</svg>'''
favicon_uri = "data:image/svg+xml;base64," + base64.b64encode(_svg.encode()).decode()

# ── Full corpus charts + global per-field xmax ────────────────────────────
fc_charts_local = {}
field_xmax      = {}

for field in DETAIL_FIELDS:
    fc_df = insight_charts.get("full_corpus", {}).get(field)
    if fc_df is None or fc_df.empty:
        continue
    labels, values = df_to_chart(fc_df, field)
    fc_charts_local[field] = {"labels": labels, "values": values}
    field_xmax[field] = max(values) * 1.2 if values else 1.0

# ── Comprehensive corpus category baseline ───────────────────────────────────
# Uses all categories (not just top 10) so niche categories like Candy, Chips,
# Party Favors still get a baseline tick showing their share in the full corpus.
if not resource_category_df.empty:
    _corp_total = resource_category_df["quantity_count"].sum()
    _corpus_cat_map = (
        resource_category_df
        .groupby("item_category", dropna=True)["quantity_count"]
        .sum()
        .div(_corp_total)
        .round(6)
        .to_dict()
    )
else:
    _corpus_cat_map = {}

for iid, charts in insight_charts.items():
    if iid in ("full_corpus", "total_sample"):
        continue
    for field in DETAIL_FIELDS:
        ch = charts.get(field)
        if ch is None or (hasattr(ch, "empty") and ch.empty):
            continue
        _, values = df_to_chart(ch, field)
        candidate = max(values) * 1.2 if values else 0.0
        if candidate > field_xmax.get(field, 0):
            field_xmax[field] = candidate

for field in field_xmax:
    field_xmax[field] = round(field_xmax[field], 4)
    fc_charts_local.setdefault(field, {})["xmax"] = field_xmax[field]

# ── Extract prose body text ───────────────────────────────────────────────
insight_bodies = {}
_walk(structured, insight_bodies)

# ── Item names: insight_id → [[name, qty], ...] ───────────────────────────
# Stored as pairs so the template can render both name and quantity.
item_names_lookup = (
    chart_ready_items_df
    .sort_values("rank")
    .groupby("insight_id")
    .apply(lambda g: list(zip(g["item_name"], g["quantity_count"].astype(int))))
    .to_dict()
)

# ── TVD lookup ────────────────────────────────────────────────────────────
tvd_lookup = {row["insight_id"]: row.to_dict() for _, row in distinctness_df.iterrows()}

# ── Filter description from CFG ───────────────────────────────────────────
try:
    _filters_list  = CFG.get("analysis", {}).get("filters", [])
    _filter_logic  = CFG.get("analysis", {}).get("filter_logic", "and").upper()
    _lenses_label = ", ".join(f.replace("_", " ") for f in ALL_GROUPBY_FIELDS)
    if _filters_list:
        _filter_str     = f" {_filter_logic} ".join(_fmt_filter(f) for f in _filters_list)
        filter_headline = (f"Insights were found by deep diving on {_lenses_label}, "
                           f"filtered on {_filter_str}")
        filter_detail   = (f"{len(_filters_list)} filter{'s' if len(_filters_list) != 1 else ''} "
                           f"applied ({_filter_logic}) · {len(curated_df)} insights accepted "
                           f"from {len(df):,} projects")
    else:
        filter_headline = (f"Insights were found by deep diving on {_lenses_label} "
                           f"across {len(df):,} projects")
        filter_detail   = f"No filters applied · {len(curated_df)} insights accepted"
except Exception as e:
    filter_headline = f"Insights were found by deep diving on {GROUPBY_FIELD.replace('_',' ')}"
    filter_detail   = f"No filters applied · {len(curated_df)} insights accepted"
    print(f"Warning: could not read filter config ({e})")

# ── Insight tiers (from manual ranking CSVs) ──────────────────────────────
_tier_lookup = _load_insight_tiers(RANKING_FILES)

# ── Group tabs: strategic_area groups first (alphabetical), then non_strategic (alphabetical) ──
_grp = (curated_df[["source_parent", "source_run_type"]].astype(str)
        .drop_duplicates(subset="source_parent"))
_grp["_label"] = _grp["source_parent"].map(_display_name)
_grp["_rank"]  = _grp["source_run_type"].map({"strategic_area": 0, "non_strategic": 1}).fillna(2).astype(int)
_grp = _grp.sort_values(["_rank", "_label"], kind="stable")
group_tabs = [{"key": k, "label": lbl} for k, lbl in zip(_grp["source_parent"], _grp["_label"])]

# ── Build insights list ───────────────────────────────────────────────────
insights_out = []

for _, row in curated_df.sort_values("supporting_project_count", ascending=False).iterrows():
    iid      = row["insight_id"]
    # Card DOM id. Prefer global_insight_id so Ask Compass can find the card;
    # fall back to the NB05 id for insights outside the agent snapshot.
    # Every internal join below still uses iid.
    card_id  = str(row.get("global_insight_id") or "") or iid
    body     = insight_bodies.get(iid, {"finding": "", "evidence_basis": "", "scope_or_caveat": "", "why_it_matters": ""})
    tvd_r    = tvd_lookup.get(iid, {})
    is_cross = _is_cross(row)

    _src       = str(row.get("source_run_type", ""))
    _parent    = str(row.get("source_parent", ""))
    _title_key = str(row.get("title", "")).strip().casefold()
    _tier      = _tier_lookup.get((_src, _title_key), "")

    # Build per-field chart dicts with labels, values, baseline, xmax
    charts_out = {}
    for field in DETAIL_FIELDS:
        ic = insight_charts.get(iid, {}).get(field)
        if ic is None or (hasattr(ic, "empty") and ic.empty):
            continue
        labels, values = df_to_chart(ic, field)
        if not labels:
            continue
        fc_lbls = fc_charts_local.get(field, {}).get("labels", labels)
        fc_vals = fc_charts_local.get(field, {}).get("values", [0.0] * len(labels))
        fc_map  = dict(zip(fc_lbls, fc_vals))
        charts_out[field] = {
            "labels":   labels,
            "values":   values,
            "baseline": [_corpus_cat_map.get(lbl, 0.0) if field == "item_category" else fc_map.get(lbl) for lbl in labels],
            "xmax":     field_xmax.get(field, 1.0),
        }

    seps = compute_separators(charts_out, fc_charts_local)

    # category_bucket is now "groupby_field::group_val" — split for display
    _bucket_raw   = str(row.get("category_bucket", ""))
    _bucket_parts = _bucket_raw.split("::", 1)
    _bucket_display = _bucket_parts[1] if len(_bucket_parts) == 2 else _bucket_raw
    
    insights_out.append({
        "id":                   card_id,
        "nb05_insight_id":      iid,
        "title":                str(row.get("title", row.get("theme", iid))),
        "finding":              body["finding"],
        "evidence_basis":       body["evidence_basis"],
        "scope_or_caveat":      body["scope_or_caveat"],
        "why_it_matters":       body["why_it_matters"],
        "tier":                 _tier,
        "group_key":            _parent,
        "group_display":        _display_name(_parent),
        "category_bucket":      _bucket_display,   # raw group value, e.g. "Books"
        "lens":                 str(row.get("source_groupby_field", GROUPBY_FIELD)),
        "source_run_id":        str(row.get("source_run_id", RUN_ID)),
        "is_cross":             is_cross,
        "section":              str(row.get("section", "")),
        "report_section":       str(row.get("report_section", "")),
        "project_count":        int(_safe_float(row.get("supporting_project_count", 0))),
        "pct_of_total":         round(
            _safe_float(row.get("supporting_project_count", 0)) / max(ANALYSIS_N, 1), 4
        ),
        "verified_topic_count": int(_safe_float(row.get("verified_topic_count", 0))),
        "mean_topic_share":     round(_safe_float(
            row.get("mean_topic_share_all_verified_topics", 0)), 3),
        "verification_ratio":   round(_safe_float(row.get("verification_ratio", 0)), 3),
        "looker_url":           str(row.get("looker_url", "")),
        "sep1_val":             seps["sep1_val"],
        "sep1_lbl":             seps["sep1_lbl"],
        "sep2_val":             seps["sep2_val"],
        "sep2_lbl":             seps["sep2_lbl"],
        "item_names":           item_names_lookup.get(iid, []),
        "tvd": {
            "metro": round(_safe_float(tvd_r.get("metro", 0)), 4),
            "grade": round(_safe_float(tvd_r.get("grade", 0)), 4),
            "efs":   round(_safe_float(tvd_r.get("efs",   0)), 4),
            "race":  round(_safe_float(tvd_r.get("race",  0)), 4),
            "mean":  round(_safe_float(tvd_r.get("mean_tvd", 0)), 4),
        },
        "charts": charts_out,
    })

# Cross-category first, then main-section, then by project count descending
_report_order_lookup = {
    row["insight_id"]: int(row.get("report_order", 9999))
    for _, row in curated_df.iterrows()
}
insights_out.sort(key=lambda x: (
    0 if x["is_cross"] else 1,
    0 if "main" in x["report_section"].lower() else 1,
    _report_order_lookup.get(x["id"], 9999),
))

# ── Account metadata ──────────────────────────────────────────────────────
account_meta_path = ROOT / "account_meta.json"
account_meta = (
    json.loads(account_meta_path.read_text())
    if account_meta_path.exists()
    else {"name": "", "logo": "", "date": RUN_DATE}
)

# ── Assemble JSON payload ─────────────────────────────────────────────────
payload = {
    "meta": {
        "run_id":           RUN_ID,
        "run_date":         RUN_DATE,
        "project_count":    int(df["project_id"].nunique()),
        "insight_count":    len(curated_df),
        "groupby_field":    GROUPBY_FIELD,         # primary field; kept for single-run template compat
        "groupby_fields":   ALL_GROUPBY_FIELDS,    # full list for multi-run templates
        "filter_headline":  filter_headline,
        "filter_detail":    filter_detail,
        "filter_range":     _filter_range_str(_filters_list),
        "group_tabs":       group_tabs,
        "looker_id_limit":  int(CFG.get("output", {}).get("looker_id_limit", 500)),
    },
    "account":     account_meta,
    "full_corpus": fc_charts_local,
    "insights":    insights_out,
}

# ── Read the latest template + Chart.js, inject payload, write HTML ──────────────────
chartjs_path  = ROOT / "chart.umd.min.js"

def _template_version(p):
    """Return (major, minor) from report_template_vX.Y.html; (0, 0) if unversioned."""
    m = _re.search(r'_v(\d+)[\._](\d+)', p.stem)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

_templates = sorted(ROOT.glob("report_template*.html"), key=_template_version)
if not _templates:
    raise FileNotFoundError("No report_template*.html found in ROOT")
template_path = _templates[-1]
print(f"Using template: {template_path.name}")

if not template_path.exists():
    raise FileNotFoundError(f"report_template.html not found at {template_path}")
if not chartjs_path.exists():
    raise FileNotFoundError(
        f"chart.umd.min.js not found at {chartjs_path}\n"
        "Download from cdn.jsdelivr.net/npm/chart.js/dist/chart.umd.min.js"
    )

template = template_path.read_text(encoding="utf-8")
chartjs  = chartjs_path.read_text(encoding="utf-8")

html_out = (
    template
    .replace("__CHARTJS__",     chartjs)
    .replace("__FAVICON__",     favicon_uri)
    .replace("__REPORT_DATA__", json.dumps(payload, ensure_ascii=False, separators=(",",":")))
)

_ts = pd.Timestamp.now().strftime("%y%m%d_%H%M")
html_path = OUT("reports", f"classroom_compass_{_ts}.html")
html_path.write_text(html_out, encoding="utf-8")

# ── Summary ───────────────────────────────────────────────────────────────
size_kb  = len(html_out.encode("utf-8")) / 1024
body_ct  = sum(1 for i in insights_out if i["finding"])
sep_ct   = sum(1 for i in insights_out if i["sep1_val"] is not None)
cross_ct = sum(1 for i in insights_out if i["is_cross"])
tier_ct  = sum(1 for i in insights_out if i["tier"])

print(f"HTML report → {html_path}")
print(f"Size: {size_kb:.0f} KB  ·  Insights: {len(insights_out)}  ·  Projects: {len(df):,}")
print(f"Insight tiers matched: {tier_ct}/{len(insights_out)}")
if tier_ct < len(insights_out):
    _miss = [(i["group_key"], i["title"]) for i in insights_out if not i["tier"]][:8]
    for gk, ti in _miss:
        print(f"   [no tier] {gk} :: {ti[:70]}")
print(f"Body text: {body_ct}/{len(insights_out)}  ·  "
      f"Separators: {sep_ct}/{len(insights_out)}  ·  "
      f"Cross-category: {cross_ct}")
print(f"Filter: {filter_headline[:120]}")

/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_20262/1165277966.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["item_name"], g["quantity_count"].astype(int))))


Using template: report_template_v2.2.html
HTML report → OUTPUTS/runs/non_strategic/project_category_bucketed/20260524_141500_project_category_bucketed_0c79fb23/reports/classroom_compass_260924_1009.html
Size: 10346 KB  ·  Insights: 1052  ·  Projects: 818,564
Insight tiers matched: 1028/1052
   [no tier] safety_justice :: Ordinary supplies become protective when they reduce fear or dysregula
   [no tier] district_name :: Communication therapy should not be merged with multilingual support
   [no tier] safety_justice :: Peer roles create protective daily structure
   [no tier] district_name :: Seating and storage are classroom workflow tools
   [no tier] safety_justice :: Attendance rewards are bundled with day-stabilizing supports
   [no tier] district_name :: Choice reading grows through high-demand titles
   [no tier] district_name :: Phonics intervention is separate from library motivation
   [no tier] safety_justice :: Attendance paperwork serves coordination, not reengagement
Body 